# 01 — Team-Level Feature Engineering

Extracts **one row per (tournament_id, team_id)** containing all features that
depend only on the team and the tournament — not on the specific opponent.

| Feature group | Source |
|---|---|
| Historical WC performance | `worldcup.db` → `team_appearances` |
| Squad demographics & experience | `worldcup.db` → `squads`, `players` |
| Coach attributes | `worldcup.db` → `manager_appointments`, `managers` |
| Elo rating | `data/raw/elo_ratings.csv` |
| FIFA ranking | `data/raw/fifa_ranking-2024-06-20.csv` |

**Output:** `./data/features_team.parquet`  
**Window:** all men's WC tournaments in the DB (no year filter here; the model
notebook controls the training window).

> Match-specific features that require knowing the opponent (confederation
> win-rate vs opponent, rest-days diff, is-host-nation, same-confederation)
> are built in `03_model_training.ipynb`.

In [31]:
import warnings

warnings.filterwarnings("ignore")

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────────────────────
# This notebook lives at notebooks/model/multiclass/v2_integration/
BASE = Path("../../../../")  # repo root
DB_PATH = BASE / "data/raw/worldcup/data-sqlite/worldcup.db"
ELO_PATH = BASE / "data/raw/elo_ratings.csv"
FIFA_PATH = BASE / "data/raw/fifa_ranking-2024-06-20.csv"
OUT_DIR = Path("./data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DB_PATH.is_file(), f"DB not found: {DB_PATH}"
assert ELO_PATH.is_file(), f"Elo file not found: {ELO_PATH}"
assert FIFA_PATH.is_file(), f"FIFA ranking not found: {FIFA_PATH}"
print("Paths OK")

Paths OK


## 1 — Load raw tables from DB

In [32]:
con = sqlite3.connect(DB_PATH)


def _sql(q):
    return pd.read_sql_query(q, con)


matches = _sql("SELECT * FROM matches")
tournaments = _sql("SELECT * FROM tournaments")
teams = _sql("SELECT * FROM teams")
team_appearances = _sql("SELECT * FROM team_appearances")
squads = _sql("SELECT * FROM squads")
players = _sql("SELECT * FROM players")
manager_appointments = _sql("SELECT * FROM manager_appointments")
managers = _sql("SELECT * FROM managers")
manager_appearances = _sql("SELECT * FROM manager_appearances")
con.close()

# Parse dates
tournaments["start_dt"] = pd.to_datetime(
    tournaments["start_date"], unit="D", origin="unix", errors="coerce"
)
matches["match_dt"] = pd.to_datetime(
    matches["match_date"], unit="D", origin="unix", errors="coerce"
)
tourn_meta = tournaments[
    ["tournament_id", "tournament_name", "year", "host_country", "start_dt"]
].copy()

# Keep men's WC only: both men's and women's use "WC-" prefix in this DB,
# so exclude any tournament with "Women" in the name
men_ids = tourn_meta.loc[
    ~tourn_meta["tournament_name"].str.contains("Women", case=False, na=False),
    "tournament_id",
].unique()
matches = matches[matches["tournament_id"].isin(men_ids)].copy()
team_appearances = team_appearances[
    team_appearances["tournament_id"].isin(men_ids)
].copy()

ta_hist = team_appearances.merge(
    tourn_meta[["tournament_id", "year"]], on="tournament_id", how="left"
).merge(
    matches[["match_id", "knockout_stage", "group_stage"]],
    on="match_id",
    how="left",
    suffixes=("", "_m"),
)

# All team×tournament combos in men's WC
team_tourn = (
    team_appearances[["tournament_id", "team_id"]]
    .drop_duplicates()
    .merge(tourn_meta[["tournament_id", "year"]], on="tournament_id", how="left")
)

print(f"matches         : {matches.shape}")
print(f"team_appearances: {team_appearances.shape}")
print(f"team×tournament : {team_tourn.shape}")
print(f"years           : {sorted(team_tourn['year'].dropna().unique().astype(int))}")

matches         : (964, 29)
team_appearances: (1928, 17)
team×tournament : (489, 3)
years           : [np.int64(1930), np.int64(1934), np.int64(1938), np.int64(1950), np.int64(1954), np.int64(1958), np.int64(1962), np.int64(1966), np.int64(1970), np.int64(1974), np.int64(1978), np.int64(1982), np.int64(1986), np.int64(1990), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]


## 2 — Historical team performance (pre-tournament, men's WC only)

For each team × tournament we compute cumulative statistics from **all prior
men's WCs**. Win rates use Laplace smoothing `(wins + α) / (n + α + β)` with
α = β = 2 (same as v1 baseline).

In [33]:
ALPHA, BETA = 2.0, 2.0


def shrink(w, n, a=ALPHA, b=BETA):
    return (w + a) / (n + a + b)


def prior_team_stats(team_id, before_year):
    sub = ta_hist[(ta_hist["team_id"] == team_id) & (ta_hist["year"] < before_year)]
    if sub.empty:
        return dict(
            hist_n_tournaments=0,
            hist_n_matches=0,
            hist_wins=0,
            hist_draws=0,
            hist_losses=0,
            hist_goal_diff_sum=0,
            hist_ko_matches=0,
            hist_ko_wins=0,
            hist_pso_matches=0,
            hist_pso_wins=0,
            hist_et_matches=0,
            hist_tournaments_with_ko_game=0,
        )
    ko = sub[sub["knockout_stage"] == 1]
    pso = sub[sub["penalty_shootout"] == 1]
    return dict(
        hist_n_tournaments=sub["tournament_id"].nunique(),
        hist_n_matches=len(sub),
        hist_wins=int(sub["win"].sum()),
        hist_draws=int(sub["draw"].sum()),
        hist_losses=int(sub["lose"].sum()),
        hist_goal_diff_sum=float((sub["goals_for"] - sub["goals_against"]).sum()),
        hist_ko_matches=len(ko),
        hist_ko_wins=int(ko["win"].sum()),
        hist_pso_matches=len(pso),
        hist_pso_wins=int((pso["penalties_for"] > pso["penalties_against"]).sum()),
        hist_et_matches=int(sub["extra_time"].sum()),
        hist_tournaments_with_ko_game=len(set(ko["tournament_id"].unique())),
    )


# Build once for all (team_id, year) combos
years_all = sorted(team_tourn["year"].dropna().unique())
all_tids = team_tourn["team_id"].unique()
hist_cache = {
    (tid, int(y)): prior_team_stats(tid, int(y)) for tid in all_tids for y in years_all
}

hist_rows = []
for _, row in team_tourn.iterrows():
    key = (row["team_id"], int(row["year"]))
    h = hist_cache[key]
    n, w = h["hist_n_matches"], h["hist_wins"]
    km, kw = h["hist_ko_matches"], h["hist_ko_wins"]
    pm, pw = h["hist_pso_matches"], h["hist_pso_wins"]
    hist_rows.append(
        {
            "tournament_id": row["tournament_id"],
            "team_id": row["team_id"],
            "year": row["year"],
            **h,
            "hist_win_rate_shrunk": shrink(w, n),
            "hist_draw_rate_shrunk": shrink(h["hist_draws"], n),
            "hist_goal_diff_per_match": (
                h["hist_goal_diff_sum"] / n if n > 0 else np.nan
            ),
            "hist_ko_win_rate_shrunk": shrink(kw, km),
            "hist_pso_win_rate_shrunk": shrink(pw, pm) if pm > 0 else np.nan,
            "hist_et_rate": h["hist_et_matches"] / n if n > 0 else np.nan,
            "hist_frac_ko": (
                h["hist_tournaments_with_ko_game"] / h["hist_n_tournaments"]
                if h["hist_n_tournaments"] > 0
                else np.nan
            ),
        }
    )

hist_df = pd.DataFrame(hist_rows)
print(f"hist_df shape: {hist_df.shape}")

hist_df shape: (489, 22)


## 3 — Squad demographics & WC experience

In [34]:
sq = squads.merge(tourn_meta[["tournament_id", "year", "start_dt"]], on="tournament_id")
sq = sq[sq["tournament_id"].isin(men_ids)].copy()
sq = sq.merge(players, on="player_id", how="left")
sq["birth_dt"] = pd.to_datetime(sq["birth_date"], errors="coerce")
sq["age_at_tournament"] = (sq["start_dt"] - sq["birth_dt"]).dt.days / 365.25
for col in ["goal_keeper", "defender", "midfielder", "forward"]:
    sq[col] = sq[col].fillna(0).astype(int)

# Prior WC appearances per player (men's only, before this tournament)
pyt = sq[["player_id", "tournament_id", "year"]].drop_duplicates()
prior_rows = []
for (pid, y), g in pyt.groupby(["player_id", "year"]):
    n_prior = pyt[(pyt["player_id"] == pid) & (pyt["year"] < y)][
        "tournament_id"
    ].nunique()
    prior_rows.append({"player_id": pid, "year": y, "prior_wc_played": n_prior})
sq = sq.merge(pd.DataFrame(prior_rows), on=["player_id", "year"], how="left")

# Aggregate to team×tournament
squad_records = []
for (tid, tmid), g in sq.groupby(["tournament_id", "team_id"]):
    squad_records.append(
        dict(
            tournament_id=tid,
            team_id=tmid,
            squad_n_players=len(g),
            squad_age_mean=g["age_at_tournament"].mean(),
            squad_age_median=g["age_at_tournament"].median(),
            squad_age_std=g["age_at_tournament"].std(),
            squad_age_min=g["age_at_tournament"].min(),
            squad_age_max=g["age_at_tournament"].max(),
            squad_n_gk=int(g["goal_keeper"].sum()),
            squad_n_df=int(g["defender"].sum()),
            squad_n_mf=int(g["midfielder"].sum()),
            squad_n_fw=int(g["forward"].sum()),
            squad_prior_wc_mean=g["prior_wc_played"].mean(),
            squad_prior_wc_median=g["prior_wc_played"].median(),
            squad_share_any_prior_wc=(g["prior_wc_played"] > 0).mean(),
            squad_share_ge2_prior_wc=(g["prior_wc_played"] >= 2).mean(),
        )
    )
squad_df = pd.DataFrame(squad_records)

# Squad continuity: Jaccard overlap with previous WC squad (same team)
overlap_rows = []
for team_id, g in sq.groupby("team_id"):
    g = g.sort_values("year")
    years_order = sorted(g["year"].unique())
    prev_players = None
    for y in years_order:
        cur = set(g.loc[g["year"] == y, "player_id"].unique())
        tid = g.loc[g["year"] == y, "tournament_id"].iloc[0]
        if prev_players is not None:
            inter = len(cur & prev_players)
            uni = len(cur | prev_players)
            overlap_rows.append(
                dict(
                    tournament_id=tid,
                    team_id=team_id,
                    squad_jaccard_vs_prev_wc=inter / uni if uni else np.nan,
                    squad_overlap_count_vs_prev_wc=inter,
                )
            )
        prev_players = cur
overlap_df = pd.DataFrame(overlap_rows)

squad_df = squad_df.merge(overlap_df, on=["tournament_id", "team_id"], how="left")
print(f"squad_df shape: {squad_df.shape}")

squad_df shape: (489, 18)


## 4 — Coach / manager features

In [35]:
ma = manager_appointments.merge(
    tourn_meta[["tournament_id", "year"]], on="tournament_id"
).merge(managers, on="manager_id", how="left")
ma = ma[ma["tournament_id"].isin(men_ids)].copy()
# One manager per team per tournament (keep first)
ma_pick = ma.sort_values("manager_id").drop_duplicates(
    subset=["tournament_id", "team_id"], keep="first"
)


def _norm(s):
    return s.astype(str).str.strip().str.lower().str.replace(" ", "", regex=False)


# Is the manager from the same country as the team?
team_name_map = teams.set_index("team_id")["team_name"].to_dict()
ma_pick["team_name"] = ma_pick["team_id"].map(team_name_map)
ma_pick["manager_local"] = (
    _norm(ma_pick["country_name"]) == _norm(ma_pick["team_name"])
).astype(int)

# Prior WC count per manager-team combo (at each tournament)
mgr_tourneys = (
    ma_pick[["manager_id", "tournament_id", "year"]]
    .drop_duplicates()
    .sort_values(["manager_id", "year"])
)
mgr_prior_rows = []
for mid, g in mgr_tourneys.groupby("manager_id"):
    for i, (_, r) in enumerate(g.iterrows()):
        mgr_prior_rows.append(
            {
                "manager_id": mid,
                "tournament_id": r["tournament_id"],
                "mgr_n_prior_wc": i,
            }
        )
mgr_prior_df = pd.DataFrame(mgr_prior_rows)

# Consecutive tournaments with same team
streak_rows = []
for (mid, tid), g in ma_pick.sort_values("year").groupby(["manager_id", "team_id"]):
    g = g.drop_duplicates(subset=["tournament_id", "year"]).sort_values("year")
    yrs = g["year"].astype(int).tolist()
    tids = g["tournament_id"].tolist()
    run = 0
    for i, y in enumerate(yrs):
        run = (run + 1) if (i == 0 or y - yrs[i - 1] in (1, 4)) else 1
        streak_rows.append(
            {
                "manager_id": mid,
                "team_id": tid,
                "tournament_id": tids[i],
                "mgr_consecutive_wc_with_team": run,
            }
        )
streak_df = pd.DataFrame(streak_rows)

# Historical win rate per manager (before this tournament)
mapp = manager_appearances.merge(
    tourn_meta[["tournament_id", "year"]], on="tournament_id"
).merge(
    team_appearances[["tournament_id", "match_id", "team_id", "win"]],
    on=["tournament_id", "match_id", "team_id"],
    how="left",
)
mapp = mapp[mapp["tournament_id"].isin(men_ids)].copy()

mgr_hist_rows = []
for mid, g in mapp.groupby("manager_id"):
    g = g.sort_values("year")
    for y in g["year"].unique():
        sub = g[(g["year"] < y)].dropna(subset=["win"])
        mgr_hist_rows.append(
            {
                "manager_id": mid,
                "year": y,
                "mgr_hist_win_rate_shrunk": shrink(int(sub["win"].sum()), len(sub)),
            }
        )
mgr_hist_df = pd.DataFrame(mgr_hist_rows).drop_duplicates(["manager_id", "year"])

# Merge all coach features
coach_df = (
    ma_pick[["tournament_id", "team_id", "manager_id", "manager_local", "year"]]
    .merge(mgr_prior_df, on=["manager_id", "tournament_id"], how="left")
    .merge(streak_df, on=["manager_id", "team_id", "tournament_id"], how="left")
    .merge(mgr_hist_df, on=["manager_id", "year"], how="left")
    .drop(columns=["manager_id", "year"])
)
print(f"coach_df shape: {coach_df.shape}")

coach_df shape: (489, 6)


## 5 — Elo rating (pre-tournament)

In [36]:
elo_raw = pd.read_csv(ELO_PATH)

# Map team names: elo_ratings uses English names, DB uses team_name
# Use the team_name column in teams table
team_names = teams[["team_id", "team_name", "team_code"]].copy()

# Elo: each row is (year, team, rating). We want the rating in the year
# immediately BEFORE each WC (i.e., the rating for that calendar year).
# Following v1: use elo year == WC year (ratings are published at year-start,
# so the WC-year rating is the one available entering the tournament).
elo_wc = elo_raw.rename(
    columns={"team": "elo_team_name", "rating": "elo_rating", "rank": "elo_rank"}
)


# Fuzzy-match team names: build a lookup from elo_team_name → team_id
# We use the same approach as v1: merge on normalized team names
def _norm_str(s):
    return str(s).strip().lower().replace(" ", "")


elo_wc["_name_norm"] = elo_wc["elo_team_name"].apply(_norm_str)
team_names["_name_norm"] = team_names["team_name"].apply(_norm_str)

# Build WC-year Elo table per team_id
wc_years = sorted(team_tourn["year"].dropna().unique().astype(int))
elo_lookup = elo_wc[
    ["year", "elo_team_name", "_name_norm", "elo_rating", "elo_rank"]
].copy()
elo_lookup["year"] = elo_lookup["year"].astype(int)

elo_rows = []
for _, row in team_tourn.iterrows():
    y = int(row["year"])
    tid = row["team_id"]
    tname_norm = team_names.loc[team_names["team_id"] == tid, "_name_norm"].values
    if len(tname_norm) == 0:
        elo_rows.append(
            {
                "tournament_id": row["tournament_id"],
                "team_id": tid,
                "elo_rating": np.nan,
                "elo_rank": np.nan,
            }
        )
        continue
    tname_norm = tname_norm[0]
    # Exact match first
    match = elo_lookup[
        (elo_lookup["year"] == y) & (elo_lookup["_name_norm"] == tname_norm)
    ]
    if match.empty:
        # Try year-1 (some tables published prior year)
        match = elo_lookup[
            (elo_lookup["year"] == y - 1) & (elo_lookup["_name_norm"] == tname_norm)
        ]
    if not match.empty:
        elo_rows.append(
            {
                "tournament_id": row["tournament_id"],
                "team_id": tid,
                "elo_rating": match.iloc[0]["elo_rating"],
                "elo_rank": match.iloc[0]["elo_rank"],
            }
        )
    else:
        elo_rows.append(
            {
                "tournament_id": row["tournament_id"],
                "team_id": tid,
                "elo_rating": np.nan,
                "elo_rank": np.nan,
            }
        )

elo_df = pd.DataFrame(elo_rows)
elo_coverage = elo_df["elo_rating"].notna().mean()
print(f"Elo coverage: {elo_coverage:.1%}")
print(f"elo_df shape: {elo_df.shape}")

Elo coverage: 99.2%
elo_df shape: (489, 4)


## 6 — FIFA ranking (pre-tournament snapshot)

In [37]:
fifa_raw = pd.read_csv(FIFA_PATH, parse_dates=["rank_date"])

# FIFA official names differ from DB team_name for several countries.
# Map FIFA name → DB team_name (both sides will be normalised later).
FIFA_NAME_MAP = {
    # Keys and values must be post-_norm_str form (no spaces, lowercase).
    # _norm_str does: strip → lower → remove spaces.
    "iriran": "iran",  # FIFA 'IR Iran'        → DB 'Iran'
    "korearepublic": "southkorea",  # FIFA 'Korea Republic' → DB 'South Korea'
    "koreadpr": "northkorea",  # FIFA 'Korea DPR'      → DB 'North Korea'
    "usa": "unitedstates",  # FIFA 'USA'            → DB 'United States'
    "côted'ivoire": "ivorycoast",  # FIFA 'Côte d'Ivoire'  → DB 'Ivory Coast'
    "coted'ivoire": "ivorycoast",  # FIFA 'Cote d'Ivoire'  → DB 'Ivory Coast'
    "czechia": "czechrepublic",  # FIFA newer 'Czechia'  → DB 'Czech Republic'
    "türkiye": "turkey",  # FIFA newer 'Türkiye'  → DB 'Turkey'
    "turkiye": "turkey",
    "chinapr": "china",  # FIFA 'China PR'       → DB 'China'
    "trinidadtobago": "trinidadandtobago",  # FIFA 'Trinidad & Tobago'
    "trinidad&tobago": "trinidadandtobago",
    "northmacedonia": "northmacedonia",  # already same, keep for clarity
}


def _fifa_norm(s):
    """Normalise then apply FIFA→DB name mapping."""
    n = _norm_str(s)
    return FIFA_NAME_MAP.get(n, n)


fifa_raw["_name_norm"] = fifa_raw["country_full"].apply(_fifa_norm)

# WC start dates (approximate) for pre-tournament snapshot
WC_START = {
    1994: "1994-06-17",
    1998: "1998-06-10",
    2002: "2002-05-31",
    2006: "2006-06-09",
    2010: "2010-06-11",
    2014: "2014-06-12",
    2018: "2018-06-14",
    2022: "2022-11-20",
}
WC_START = {k: pd.Timestamp(v) for k, v in WC_START.items()}

fifa_rows = []
for _, row in team_tourn.iterrows():
    y = int(row["year"])
    tid = row["team_id"]
    cutoff = WC_START.get(y)
    if cutoff is None:
        fifa_rows.append(
            {
                "tournament_id": row["tournament_id"],
                "team_id": tid,
                "fifa_rank": np.nan,
                "fifa_points": np.nan,
            }
        )
        continue
    tname_norm = team_names.loc[team_names["team_id"] == tid, "_name_norm"].values
    if len(tname_norm) == 0:
        fifa_rows.append(
            {
                "tournament_id": row["tournament_id"],
                "team_id": tid,
                "fifa_rank": np.nan,
                "fifa_points": np.nan,
            }
        )
        continue
    tname_norm = tname_norm[0]
    snap = fifa_raw[
        (fifa_raw["rank_date"] <= cutoff) & (fifa_raw["_name_norm"] == tname_norm)
    ]

    if snap.empty:
        fifa_rows.append(
            {
                "tournament_id": row["tournament_id"],
                "team_id": tid,
                "fifa_rank": np.nan,
                "fifa_points": np.nan,
            }
        )
    else:
        best = snap.sort_values("rank_date").iloc[-1]
        fifa_rows.append(
            {
                "tournament_id": row["tournament_id"],
                "team_id": tid,
                "fifa_rank": best["rank"],
                "fifa_points": best["total_points"],
            }
        )

fifa_df = pd.DataFrame(fifa_rows)
fifa_coverage = fifa_df["fifa_rank"].notna().mean()
print(f"FIFA coverage: {fifa_coverage:.1%}")
print(f"fifa_df shape: {fifa_df.shape}")

# Diagnose: list teams that couldn't be matched in FIFA-covered years (1994+)
covered_years = set(WC_START.keys())
miss_fifa = fifa_df.merge(team_tourn, on=["tournament_id", "team_id"]).merge(
    teams[["team_id", "team_name"]], on="team_id", how="left"
)
miss_fifa = (
    miss_fifa[miss_fifa["year"].isin(covered_years) & miss_fifa["fifa_rank"].isna()][
        ["team_name", "year"]
    ]
    .drop_duplicates()
    .sort_values(["year", "team_name"])
)
if len(miss_fifa) > 0:
    print(f"\nTeams missing FIFA rank in covered years ({len(miss_fifa)} cases):")
    print(miss_fifa.to_string(index=False))
else:
    print("\nAll teams matched in FIFA-covered years ✓")

FIFA coverage: 50.7%
fifa_df shape: (489, 4)

All teams matched in FIFA-covered years ✓


## 7 — Merge all & save

In [38]:
features_team = (
    hist_df.merge(squad_df, on=["tournament_id", "team_id"], how="left")
    .merge(coach_df, on=["tournament_id", "team_id"], how="left")
    .merge(elo_df, on=["tournament_id", "team_id"], how="left")
    .merge(fifa_df, on=["tournament_id", "team_id"], how="left")
    # Add confederation for use in model notebook
    .merge(
        teams[["team_id", "confederation_id", "team_name", "team_code"]],
        on="team_id",
        how="left",
    )
)

out_path = OUT_DIR / "features_team.parquet"
features_team.to_parquet(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Shape : {features_team.shape}")
print(f"Years : {sorted(features_team['year'].dropna().unique().astype(int))}")
print(f"\nColumn list:")
for c in features_team.columns:
    print(f"  {c}")

Saved → data/features_team.parquet
Shape : (489, 49)
Years : [np.int64(1930), np.int64(1934), np.int64(1938), np.int64(1950), np.int64(1954), np.int64(1958), np.int64(1962), np.int64(1966), np.int64(1970), np.int64(1974), np.int64(1978), np.int64(1982), np.int64(1986), np.int64(1990), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]

Column list:
  tournament_id
  team_id
  year
  hist_n_tournaments
  hist_n_matches
  hist_wins
  hist_draws
  hist_losses
  hist_goal_diff_sum
  hist_ko_matches
  hist_ko_wins
  hist_pso_matches
  hist_pso_wins
  hist_et_matches
  hist_tournaments_with_ko_game
  hist_win_rate_shrunk
  hist_draw_rate_shrunk
  hist_goal_diff_per_match
  hist_ko_win_rate_shrunk
  hist_pso_win_rate_shrunk
  hist_et_rate
  hist_frac_ko
  squad_n_players
  squad_age_mean
  squad_age_median
  squad_age_std
  squad_age_min
  squad_age_max
  squad_n_gk
  squad_n_df
  squad_n_mf
  squad_n_fw
  squad_prio

In [39]:
# Quick sanity checks
print("Missing rates (top 15 by missingness):")
miss = features_team.isnull().mean().sort_values(ascending=False)
print(miss[miss > 0].head(15).to_string())
print("\nSample (year=2022, first 5 rows):")
display(
    features_team[features_team["year"] == 2022][
        [
            "team_name",
            "year",
            "elo_rating",
            "hist_win_rate_shrunk",
            "squad_age_mean",
            "fifa_rank",
        ]
    ].head()
)

Missing rates (top 15 by missingness):
hist_pso_win_rate_shrunk          0.783231
fifa_points                       0.492843
fifa_rank                         0.492843
squad_jaccard_vs_prev_wc          0.173824
hist_frac_ko                      0.173824
hist_et_rate                      0.173824
squad_overlap_count_vs_prev_wc    0.173824
hist_goal_diff_per_match          0.173824
elo_rank                          0.008180
elo_rating                        0.008180

Sample (year=2022, first 5 rows):


,team_name,year,elo_rating,hist_win_rate_shrunk,squad_age_mean,fifa_rank
457,Qatar,2022,1578.0,0.500000,27.526141,50.0
458,Ecuador,2022,1842.0,0.428571,26.067393,44.0
459,England,2022,1967.0,0.438356,26.963197,5.0
460,Iran,2022,1779.0,0.210526,29.419849,20.0
461,Senegal,2022,1747.0,0.416667,26.462170,18.0
